<a href="https://colab.research.google.com/github/tirthankarbiswas24/masai_capstone/blob/main/reconcile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook provides a reusable function reconcile_payments(ledger_df, gateway_df) that returns four DataFrames:
*   transactions missing in the gateway export
*   transactions missing in the ledger (extra in gateway)
*   amount mismatches (with the computed difference)
*   status mismatches — using set operations on transaction_id and pd.merge for the pairwise comparisons

Run it against ledger.csv vs. gateway_export.csv and report all four discrepancy counts; they must be consistent with the ~5%/~3%/~2%/~2% injection rates in generate_data.py


In [1]:
import pandas as pd

In [2]:
ledger_df = pd.read_csv("ledger.csv")
gateway_df = pd.read_csv("gateway_export.csv")
#gateway_df.describe()
#ledger_df.describe()
#ledger_df.head()

In [3]:
def reconcile_payments(ledger_df, gateway_df):
  ledger_ids = set(ledger_df['transaction_id'])
  gateway_ids = set(gateway_df['transaction_id'])
  missing_in_gateway_ids = sorted(ledger_ids - gateway_ids)
  missing_in_ledger_ids = sorted(gateway_ids - ledger_ids)
  missing_in_ledger_df = gateway_df[gateway_df['transaction_id'].isin(missing_in_ledger_ids)]
  missing_in_gateway_df = ledger_df[ledger_df['transaction_id'].isin(missing_in_gateway_ids)]
  #print("missing_in_ledger", len(missing_in_ledger_df), missing_in_ledger_df.head())
  #print("missing_in_gateway", len(missing_in_gateway_df), missing_in_gateway_df.head())

  common_ids = ledger_ids & gateway_ids
  #print("common IDs: ", len(common_ids))
  common_ledger_df = ledger_df[ledger_df['transaction_id'].isin(common_ids)]
  common_gateway_df = gateway_df[gateway_df['transaction_id'].isin(common_ids)]
  comparison_df = pd.merge(common_ledger_df, common_gateway_df, on='transaction_id', suffixes=("_gateway", "_ledger"), )
  amount_mismatch_df = comparison_df[comparison_df['amount_inr_ledger'] != comparison_df['amount_inr_gateway']].copy()
  amount_mismatch_df['amount_mismatch'] = comparison_df['amount_inr_ledger'] - comparison_df['amount_inr_gateway']
  #print("No of amount mismatches:", len(amount_mismatch_df))
  #print(amount_mismatch_df[['transaction_id', 'amount_inr_ledger', 'amount_inr_gateway', 'amount_mismatch']])

  status_mismatch_df = comparison_df[comparison_df['status_ledger'] != comparison_df['status_gateway']].copy()
  #print("No of status mismatches: ", len(status_mismatch_df))
  #print(status_mismatch_df[['transaction_id', 'amount_inr_ledger', 'status_ledger', 'status_gateway']])

  return {
     "missing_in_gateway_df": missing_in_gateway_df,
     "missing_in_gateway_count": len(missing_in_gateway_df),
     "missing_in_ledger_df": missing_in_ledger_df,
     "missing_in_ledger_count": len(missing_in_ledger_df),
     "amount_mismatch_df": amount_mismatch_df,
     "amount_mismatch_count": len(amount_mismatch_df),
     "amount_mismatch_total": amount_mismatch_df['amount_mismatch'].abs().sum(),
     "status_mismatch_df": status_mismatch_df,
     "status_mismatch_count": len(status_mismatch_df)
  }

In [ ]:
response = reconcile_payments(ledger_df, gateway_df)
print("missing_in_gateway_count: ", response['missing_in_gateway_count'])
print("missing_in_ledger_count: ", response['missing_in_ledger_count'])
print("amount_mismatch_count: ", response['amount_mismatch_count'])
print("amount_mismatch_total: ", response['amount_mismatch_total'])
print("status_mismatch_count: ", response['status_mismatch_count'])

missing_in_gateway_count:  27
missing_in_ledger_count:  10
amount_mismatch_count:  16
amount_mismatch_total:  1250
status_mismatch_count:  9
Done
